In [1]:
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

WIKIR_DIR = Path("/Users/uni/homework13April/wikIR1k")
MIRAGE_DIR = Path("/Users/uni/homework13April/mirage")
EMB_DIR    = Path("/Users/uni/homework13April/embeddings")

print("Imports OK")

Using device: mps
Imports OK


In [2]:
# Load WikIR corpus
docs_df       = pd.read_csv(WIKIR_DIR / "documents.csv")
doc_ids       = docs_df["id_right"].tolist()
doc_id_to_idx = {doc_id: idx for idx, doc_id in enumerate(doc_ids)}

# Load train queries + qrels
train_queries_df = pd.read_csv(WIKIR_DIR / "training" / "queries.csv")
train_qids   = train_queries_df["id_left"].tolist()
train_qtexts = train_queries_df["text_left"].tolist()

# Load test queries + qrels
test_queries_df = pd.read_csv(WIKIR_DIR / "test" / "queries.csv")
test_qids   = test_queries_df["id_left"].tolist()
test_qtexts = test_queries_df["text_left"].tolist()

def load_qrels(path):
    qrels = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            qid, _, docid, rel = int(parts[0]), parts[1], int(parts[2]), int(parts[3])
            if qid not in qrels:
                qrels[qid] = {}
            qrels[qid][docid] = rel
    return qrels

train_qrels = load_qrels(WIKIR_DIR / "training" / "qrels")
test_qrels  = load_qrels(WIKIR_DIR / "test"      / "qrels")

def load_bm25_res(path):
    results = defaultdict(list)
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            qid, docid, score = int(parts[0]), int(parts[2]), float(parts[4])
            results[qid].append((docid, score))
    return dict(results)

bm25_train = load_bm25_res(WIKIR_DIR / "training" / "BM25.res")
bm25_test  = load_bm25_res(WIKIR_DIR / "test"      / "BM25.res")

print(f"Train queries: {len(train_qids)}, Test queries: {len(test_qids)}")
print(f"BM25 train queries: {len(bm25_train)}, test: {len(bm25_test)}")

Train queries: 1444, Test queries: 100
BM25 train queries: 1444, test: 100


In [3]:
# Load MIRAGE
with open(MIRAGE_DIR / "dataset.json") as f:
    mirage_dataset = json.load(f)
with open(MIRAGE_DIR / "doc_pool.json") as f:
    mirage_doc_pool = json.load(f)
with open(MIRAGE_DIR / "oracle.json") as f:
    mirage_oracle = json.load(f)

mirage_docs_by_query = defaultdict(list)
for doc in mirage_doc_pool:
    mirage_docs_by_query[doc["mapped_id"]].append(doc)

print(f"MIRAGE queries: {len(mirage_dataset)}")

MIRAGE queries: 7560


In [4]:
# Evaluation functions
def precision_at_k(ranked_doc_ids, relevant_docs, k):
    hits = sum(1 for d in ranked_doc_ids[:k] if d in relevant_docs and relevant_docs[d] > 0)
    return hits / k

def ap_at_k(ranked_doc_ids, relevant_docs, k):
    hits, score = 0, 0.0
    for i, doc in enumerate(ranked_doc_ids[:k]):
        if doc in relevant_docs and relevant_docs[doc] > 0:
            hits += 1
            score += hits / (i + 1)
    num_rel = sum(1 for r in relevant_docs.values() if r > 0)
    return score / min(num_rel, k) if num_rel > 0 else 0.0

def dcg_at_k(ranked_doc_ids, relevant_docs, k):
    return sum(
        relevant_docs.get(doc, 0) / np.log2(i + 2)
        for i, doc in enumerate(ranked_doc_ids[:k])
    )

def ndcg_at_k(ranked_doc_ids, relevant_docs, k):
    actual = dcg_at_k(ranked_doc_ids, relevant_docs, k)
    ideal  = sum(r / np.log2(i + 2) for i, r in enumerate(sorted(relevant_docs.values(), reverse=True)[:k]))
    return actual / ideal if ideal > 0 else 0.0

def evaluate(results_per_query, qrels):
    p1, p10, p20, map20, ndcg20 = [], [], [], [], []
    for qid, ranked in results_per_query.items():
        rel = qrels.get(qid, {})
        p1.append(precision_at_k(ranked, rel, 1))
        p10.append(precision_at_k(ranked, rel, 10))
        p20.append(precision_at_k(ranked, rel, 20))
        map20.append(ap_at_k(ranked, rel, 20))
        ndcg20.append(ndcg_at_k(ranked, rel, 20))
    return {
        "P@1":     round(np.mean(p1), 4),
        "P@10":    round(np.mean(p10), 4),
        "P@20":    round(np.mean(p20), 4),
        "MAP@20":  round(np.mean(map20), 4),
        "nDCG@20": round(np.mean(ndcg20), 4),
    }

print("Evaluation functions ready")

Evaluation functions ready


In [5]:
def minmax_normalize(scores):
    """Min-max normalize array to [0,1]. If all values equal, return zeros."""
    mn, mx = scores.min(), scores.max()
    if mx == mn:
        return np.zeros_like(scores)
    return (scores - mn) / (mx - mn)


def compute_mixture_scores(model, model_name, doc_embs_cache, doc_id_to_idx,
                            bm25_results, query_ids, query_texts, device):
    """
    For each query, compute:
      - min-max normalized BM25 scores
      - min-max normalized cosine similarity scores
    Returns dict: {qid: (doc_ids, bm25_norm, cosine_norm)}
    """
    print(f"Encoding {len(query_texts)} queries...")
    query_embs = model.encode(
        query_texts, normalize_embeddings=True,
        convert_to_numpy=True, device=device, show_progress_bar=True
    )

    data = {}
    for i, qid in enumerate(query_ids):
        docs = bm25_results.get(qid, [])
        if not docs:
            continue
        valid_docs  = [(docid, score) for docid, score in docs if docid in doc_id_to_idx]
        if not valid_docs:
            continue
        doc_ids_q   = [docid for docid, _ in valid_docs]
        bm25_scores = np.array([score for _, score in valid_docs])
        indices     = [doc_id_to_idx[docid] for docid in doc_ids_q]
        cand_embs   = doc_embs_cache[indices]
        cos_scores  = cand_embs @ query_embs[i]

        bm25_norm = minmax_normalize(bm25_scores)
        cos_norm  = minmax_normalize(cos_scores)
        data[qid] = (doc_ids_q, bm25_norm, cos_norm)
    return data


def rank_with_alpha(data, alpha):
    """Apply mixture formula and return ranked results."""
    results = {}
    for qid, (doc_ids_q, bm25_norm, cos_norm) in data.items():
        mixture = alpha * bm25_norm + (1 - alpha) * cos_norm
        order   = np.argsort(-mixture)
        results[qid] = [doc_ids_q[j] for j in order]
    return results


def optimize_alpha(data, qrels, metric="nDCG@20", alphas=None):
    """Grid search over alpha values, return best alpha and scores."""
    if alphas is None:
        alphas = np.round(np.arange(0.0, 1.01, 0.1), 2)
    best_alpha, best_score = 0.0, -1.0
    alpha_scores = {}
    for alpha in alphas:
        results = rank_with_alpha(data, alpha)
        metrics = evaluate(results, qrels)
        alpha_scores[alpha] = metrics
        if metrics[metric] > best_score:
            best_score = metrics[metric]
            best_alpha = alpha
        print(f"  alpha={alpha:.1f}: {metric}={metrics[metric]:.4f}")
    print(f"\nBest alpha={best_alpha:.1f} with {metric}={best_score:.4f}")
    return best_alpha, alpha_scores


def mirage_mixture(model, model_name, mirage_dataset, mirage_docs_by_query,
                   mirage_oracle, device, alpha):
    """Apply mixture model to MIRAGE. BM25 computed on-the-fly."""
    results = {}
    qrels   = {}

    for entry in tqdm(mirage_dataset, desc=f"MIRAGE mixture {model_name.split('/')[-1]}"):
        qid   = entry["query_id"]
        query = entry["query"]
        candidates = mirage_docs_by_query[qid]
        if not candidates:
            continue

        chunks    = [d["doc_chunk"] for d in candidates]
        doc_names = [d["doc_name"]  for d in candidates]

        # BM25 scores
        bm25 = BM25Okapi([c.lower().split() for c in chunks])
        bm25_scores = np.array(bm25.get_scores(query.lower().split()))

        # Cosine similarity
        query_emb = model.encode(query, normalize_embeddings=True, convert_to_numpy=True)
        doc_embs  = model.encode(chunks, normalize_embeddings=True, convert_to_numpy=True)
        cos_scores = doc_embs @ query_emb

        # Normalize both
        bm25_norm = minmax_normalize(bm25_scores)
        cos_norm  = minmax_normalize(cos_scores)

        # Mixture + deduplicate by doc_name
        mixture = alpha * bm25_norm + (1 - alpha) * cos_norm
        order   = np.argsort(-mixture)
        seen, ranked_names = set(), []
        for idx in order:
            name = doc_names[idx]
            if name not in seen:
                seen.add(name)
                ranked_names.append(name)

        results[qid] = ranked_names
        relevant_name = mirage_oracle[qid]["doc_name"] if qid in mirage_oracle else None
        unique_names  = list(dict.fromkeys(doc_names))
        qrels[qid]    = {name: (1 if name == relevant_name else 0) for name in unique_names}

    return results, qrels

print("Mixture model functions ready")

Mixture model functions ready


In [6]:
## Model 1: multi-qa-MiniLM-L6-cos-v1
MODEL1_NAME = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
print(f"Loading {MODEL1_NAME}...")
model1 = SentenceTransformer(MODEL1_NAME, device=device)
safe1  = MODEL1_NAME.replace("/", "_")
doc_embs_m1 = np.load(EMB_DIR / f"wikir_docs_{safe1}.npy")
print(f"Loaded cached embeddings: {doc_embs_m1.shape}")

Loading sentence-transformers/multi-qa-MiniLM-L6-cos-v1...
Loaded cached embeddings: (369721, 384)


In [7]:
# Compute mixture scores on WikIR TRAIN set — Model 1
print("Computing train mixture scores for Model 1...")
train_data_m1 = compute_mixture_scores(
    model1, MODEL1_NAME, doc_embs_m1, doc_id_to_idx,
    bm25_train, train_qids, train_qtexts, device
)

# Optimize alpha on train set
print("\nOptimizing alpha (nDCG@20) — Model 1:")
best_alpha_m1, alpha_scores_m1 = optimize_alpha(train_data_m1, train_qrels)

Computing train mixture scores for Model 1...
Encoding 1444 queries...


Batches:   0%|          | 0/46 [00:00<?, ?it/s]


Optimizing alpha (nDCG@20) — Model 1:
  alpha=0.0: nDCG@20=0.2997
  alpha=0.1: nDCG@20=0.3062
  alpha=0.2: nDCG@20=0.3064
  alpha=0.3: nDCG@20=0.3033
  alpha=0.4: nDCG@20=0.2963
  alpha=0.5: nDCG@20=0.2838
  alpha=0.6: nDCG@20=0.2716
  alpha=0.7: nDCG@20=0.2604
  alpha=0.8: nDCG@20=0.2489
  alpha=0.9: nDCG@20=0.2389
  alpha=1.0: nDCG@20=0.2257

Best alpha=0.2 with nDCG@20=0.3064


In [8]:
# Apply best alpha to WikIR TEST set — Model 1
print(f"Applying alpha={best_alpha_m1} to WikIR test set...")
test_data_m1 = compute_mixture_scores(
    model1, MODEL1_NAME, doc_embs_m1, doc_id_to_idx,
    bm25_test, test_qids, test_qtexts, device
)
test_results_m1 = rank_with_alpha(test_data_m1, best_alpha_m1)
metrics_wikir_m1 = evaluate(test_results_m1, test_qrels)
print(f"WikIR test — Model 1 (alpha={best_alpha_m1}): {metrics_wikir_m1}")

Applying alpha=0.2 to WikIR test set...
Encoding 100 queries...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

WikIR test — Model 1 (alpha=0.2): {'P@1': 0.48, 'P@10': 0.16, 'P@20': 0.109, 'MAP@20': 0.143, 'nDCG@20': 0.3081}


In [9]:
# Apply to MIRAGE — Model 1
print(f"Applying alpha={best_alpha_m1} to MIRAGE...")
mirage_results_m1, mirage_qrels = mirage_mixture(
    model1, MODEL1_NAME, mirage_dataset, mirage_docs_by_query,
    mirage_oracle, device, best_alpha_m1
)
metrics_mirage_m1 = evaluate(mirage_results_m1, mirage_qrels)
print(f"MIRAGE — Model 1 (alpha={best_alpha_m1}): {metrics_mirage_m1}")

Applying alpha=0.2 to MIRAGE...


MIRAGE mixture multi-qa-MiniLM-L6-cos-v1: 100%|██████████| 7560/7560 [07:53<00:00, 15.97it/s]

MIRAGE — Model 1 (alpha=0.2): {'P@1': 0.8836, 'P@10': 0.1, 'P@20': 0.05, 'MAP@20': 0.9367, 'nDCG@20': 0.953}


In [10]:
import gc
del model1, doc_embs_m1
gc.collect()
print("model1 freed")

model1 freed


In [11]:
## Model 2: msmarco-distilbert-dot-v5
MODEL2_NAME = "sentence-transformers/msmarco-distilbert-dot-v5"
print(f"Loading {MODEL2_NAME}...")
model2 = SentenceTransformer(MODEL2_NAME, device=device)
safe2  = MODEL2_NAME.replace("/", "_")
doc_embs_m2 = np.load(EMB_DIR / f"wikir_docs_{safe2}.npy")
print(f"Loaded cached embeddings: {doc_embs_m2.shape}")

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 03c518bd-0deb-48b2-babd-d5e4718acbef)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/msmarco-distilbert-dot-v5/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Loading sentence-transformers/msmarco-distilbert-dot-v5...
Loaded cached embeddings: (369721, 768)


In [12]:
# Compute mixture scores on WikIR TRAIN set — Model 2
print("Computing train mixture scores for Model 2...")
train_data_m2 = compute_mixture_scores(
    model2, MODEL2_NAME, doc_embs_m2, doc_id_to_idx,
    bm25_train, train_qids, train_qtexts, device
)

# Optimize alpha on train set
print("\nOptimizing alpha (nDCG@20) — Model 2:")
best_alpha_m2, alpha_scores_m2 = optimize_alpha(train_data_m2, train_qrels)

Computing train mixture scores for Model 2...
Encoding 1444 queries...


Batches:   0%|          | 0/46 [00:00<?, ?it/s]


Optimizing alpha (nDCG@20) — Model 2:
  alpha=0.0: nDCG@20=0.2996
  alpha=0.1: nDCG@20=0.3048
  alpha=0.2: nDCG@20=0.3059
  alpha=0.3: nDCG@20=0.3028
  alpha=0.4: nDCG@20=0.2965
  alpha=0.5: nDCG@20=0.2843
  alpha=0.6: nDCG@20=0.2723
  alpha=0.7: nDCG@20=0.2606
  alpha=0.8: nDCG@20=0.2497
  alpha=0.9: nDCG@20=0.2399
  alpha=1.0: nDCG@20=0.2257

Best alpha=0.2 with nDCG@20=0.3059


In [13]:
# Apply best alpha to WikIR TEST set — Model 2
print(f"Applying alpha={best_alpha_m2} to WikIR test set...")
test_data_m2 = compute_mixture_scores(
    model2, MODEL2_NAME, doc_embs_m2, doc_id_to_idx,
    bm25_test, test_qids, test_qtexts, device
)
test_results_m2 = rank_with_alpha(test_data_m2, best_alpha_m2)
metrics_wikir_m2 = evaluate(test_results_m2, test_qrels)
print(f"WikIR test — Model 2 (alpha={best_alpha_m2}): {metrics_wikir_m2}")

Applying alpha=0.2 to WikIR test set...
Encoding 100 queries...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

WikIR test — Model 2 (alpha=0.2): {'P@1': 0.53, 'P@10': 0.158, 'P@20': 0.1075, 'MAP@20': 0.1401, 'nDCG@20': 0.3108}


In [14]:
# Apply to MIRAGE — Model 2
print(f"Applying alpha={best_alpha_m2} to MIRAGE...")
mirage_results_m2, _ = mirage_mixture(
    model2, MODEL2_NAME, mirage_dataset, mirage_docs_by_query,
    mirage_oracle, device, best_alpha_m2
)
metrics_mirage_m2 = evaluate(mirage_results_m2, mirage_qrels)
print(f"MIRAGE — Model 2 (alpha={best_alpha_m2}): {metrics_mirage_m2}")

Applying alpha=0.2 to MIRAGE...


MIRAGE mixture msmarco-distilbert-dot-v5: 100%|██████████| 7560/7560 [10:16<00:00, 12.27it/s]

MIRAGE — Model 2 (alpha=0.2): {'P@1': 0.8749, 'P@10': 0.1, 'P@20': 0.05, 'MAP@20': 0.9312, 'nDCG@20': 0.9489}


In [15]:
# Task 3 Results Table
rows = [
    {"Model": "multi-qa-MiniLM-L6-cos-v1",  "Dataset": "WikIR",  "alpha": best_alpha_m1, **metrics_wikir_m1},
    {"Model": "msmarco-distilbert-dot-v5",   "Dataset": "WikIR",  "alpha": best_alpha_m2, **metrics_wikir_m2},
    {"Model": "multi-qa-MiniLM-L6-cos-v1",  "Dataset": "MIRAGE", "alpha": best_alpha_m1, **metrics_mirage_m1},
    {"Model": "msmarco-distilbert-dot-v5",   "Dataset": "MIRAGE", "alpha": best_alpha_m2, **metrics_mirage_m2},
]
df3 = pd.DataFrame(rows)
print("=== Task 3 — Mixture Model Results ===")
print(df3.to_string(index=False))

=== Task 3 — Mixture Model Results ===
                    Model Dataset  alpha    P@1  P@10   P@20  MAP@20  nDCG@20
multi-qa-MiniLM-L6-cos-v1   WikIR    0.2 0.4800 0.160 0.1090  0.1430   0.3081
msmarco-distilbert-dot-v5   WikIR    0.2 0.5300 0.158 0.1075  0.1401   0.3108
multi-qa-MiniLM-L6-cos-v1  MIRAGE    0.2 0.8836 0.100 0.0500  0.9367   0.9530
msmarco-distilbert-dot-v5  MIRAGE    0.2 0.8749 0.100 0.0500  0.9312   0.9489
